# Group R² mixture: extent vs. strength, dyscalculia vs. control

"Vertices with R² > t" confounds **extent** (how much cortex is numerosity-tuned)
with **strength** (how well the nPRF explains the tuned vertices), and with plain
noisiness, which shifts the whole R² distribution. This notebook follows the
per-subject 2-component mixture approach worked out in
[`tutorial_r2_mixture_extent_vs_strength.md`](./tutorial_r2_mixture_extent_vs_strength.md)
(and its revision,
[`tutorial_r2_mixture_extent_vs_strength _update1.md`](./tutorial_r2_mixture_extent_vs_strength%20_update1.md)),
written for exactly this comparison (35 dyscalculics vs. 35 controls).

Per subject, with no threshold anywhere:

| Parameter | | Reads as |
|---|---|---|
| `signal_weight` | w_s | **Extent** — fraction of the ROI that is tuned |
| `signal_mu` | μ_s | **Strength** — typical logit-R² of a tuned vertex |
| `noise_mu` | μ_n | **Noise floor** — a data-quality readout (§7) |

## Scope of this notebook

Sections refer to the tutorial. Implemented:
- §3/§4: per-subject, per-ROI (`NPC`, bilateral, fixed anatomical) fit, plus a
  whole-brain fit per subject
- §5: guard rails (label-swap canary, w_s>w_n flag, separability flag) and a
  seed-stability check (`seed=0..9`)
- §6.2(a): the *free* noise anchor — pin the ROI fit's noise component to that
  subject's own whole-brain noise estimate, since that whole-brain fit is
  computed anyway
- §7: motion (`mean_fd`) and usable-run-count (`n_runs`) covariates, and
  `noise_mu ~ group` tested **before** the extent/strength tests
- §8: the group regressions, on the logit scale, via `sm.formula.ols`

**Deliberately not implemented** (see the closing cell for why): §6.2(b)
permutation-null noise anchor, §6.1 bootstrap vertex-resampling CIs, and the §7
leave-one-run-out cvR² cross-check — all three need new nPRF model fits
(`ParameterFitter`, tensorflow), not just statistics on existing R² maps.

**braincoder: `fit_r2_f_beta_mixture` is never used here.** It's the "F+Beta"
flavour from `r2_mixture_signal_voxels.ipynb`, and tutorial §5 ("the F+Beta trap")
shows it's empirically unstable on real data (w_signal swinging 20× between
subjects, FDR thresholds collapsing to 0). Only the logit-Gaussian
`fit_r2_mixture` is used.

## Setup

Same tensorflow-free import as `r2_mixture_signal_voxels.ipynb`: `braincoder.utils.__init__`
imports tensorflow, which isn't installed in the `numrefields` env, so `stats.py` is
loaded directly by path when the plain import fails. Reuses the same braincoder
install already pinned there (`feature/f-beta-r2-mixture`, commit `4acbd4b7`) —
that branch's `fit_r2_mixture` / `r2_fdr_threshold` / `plot_r2_mixture` are
identical to `main`'s (diffed to confirm), so no separate install is needed here.
`fit_r2_f_beta_mixture` exists in that install but is deliberately never imported
below (see intro).

In [10]:
import os
import os.path as op
import importlib.util

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

try:
    # normal route (works wherever tensorflow imports cleanly, e.g. the server)
    from braincoder.utils import stats as bcstats
except Exception as e:
    # tensorflow-free route: load stats.py directly, bypassing braincoder/utils/__init__.py
    print(f'plain import failed ({type(e).__name__}), loading stats.py directly')
    _bc = importlib.util.find_spec('braincoder')
    if _bc is None or not _bc.submodule_search_locations:
        raise ImportError('braincoder not importable at all -- is it installed?') from e
    _stats_fn = op.join(list(_bc.submodule_search_locations)[0], 'utils', 'stats.py')
    if not op.exists(_stats_fn):
        raise ImportError(f'{_stats_fn} not found') from e
    _spec = importlib.util.spec_from_file_location('bcstats', _stats_fn)
    bcstats = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(bcstats)

# logit-Gaussian only -- fit_r2_f_beta_mixture is intentionally not imported (see intro)
_needed = ('fit_r2_mixture', 'posterior_p_signal', 'r2_fdr_threshold', 'plot_r2_mixture')
_missing = [f for f in _needed if not hasattr(bcstats, f)]
if _missing:
    raise ImportError(
        f'braincoder is missing {_missing}. See r2_mixture_signal_voxels.ipynb intro '
        "for how to check out 'feature/f-beta-r2-mixture'.")

fit_r2_mixture      = bcstats.fit_r2_mixture
posterior_p_signal  = bcstats.posterior_p_signal
r2_fdr_threshold    = bcstats.r2_fdr_threshold
plot_r2_mixture     = bcstats.plot_r2_mixture

sns.set_style('ticks')

import braincoder
_bc_loc = list(importlib.util.find_spec('braincoder').submodule_search_locations)[0]
print(f'braincoder {braincoder.__version__} at {_bc_loc}')
print('expected pin (see r2_mixture_signal_voxels.ipynb intro): '
      'Gilles86/braincoder@feature/f-beta-r2-mixture, '
      'commit 4acbd4b79b4abe8f432e3e48223c2396384d727e')

plain import failed (ModuleNotFoundError), loading stats.py directly
braincoder 0.1.0 at /home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/braincoder
expected pin (see r2_mixture_signal_voxels.ipynb intro): Gilles86/braincoder@feature/f-beta-r2-mixture, commit 4acbd4b79b4abe8f432e3e48223c2396384d727e


## Configuration

`SPACE` is fixed to `fsaverage5` (not a toggle, unlike the exploratory notebook):
it's a shared template across subjects with a matched vertex count, which
sidesteps the fsnative vertex-area-weighting problem tutorial §4 flags ("weight by
midthickness vertex area, or `n_tuned` inherits mesh-density differences between
subjects") entirely -- no per-subject area weighting is needed here.

`N_INIT=8` EM restarts (tutorial §4/§5: sklearn's default of 1 risks different
subjects landing in different local optima, manufacturing group differences out of
optimiser noise). `SEEDS_STABILITY` re-fits each subject's ROI mixture across 10
seeds as a stability check (§5).

In [11]:
SPACE      =  'T1w' #'fsaverage5'
NPRF_KEY   = 'encoding_model.denoise'   # stim1 nPRF fits, as in r2_mixture_signal_voxels.ipynb
SESSION    = 1

ALPHA           = 0.05        # tail-FDR level
N_INIT          = 8           # EM restarts per fit
SEEDS_STABILITY = range(10)   # seed-stability check (tutorial §5)

BIDS_REMOTE = '/mnt_03/ds-dnumrisk'
BIDS_LOCAL  = '/Users/mrenke/data/ds-dnumrisk'
BIDS_shared = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'

bids_folder = BIDS_REMOTE if op.exists(BIDS_REMOTE) else BIDS_LOCAL
bids_folder_out = BIDS_shared if op.exists(BIDS_shared) else bids_folder

if bids_folder == BIDS_LOCAL:
    print(f'!! server not mounted, falling back to {BIDS_LOCAL}')
print('bids_folder:', bids_folder)

bids_folder: /mnt_03/ds-dnumrisk


## Loaders

Copied from `r2_mixture_signal_voxels.ipynb` (this codebase already duplicates this
NPC-loading pattern across `fmri_analysis/gradients/utils.py`,
`fmri_analysis/connectivity/utils_01.py` and
`fmri_analysis/parietalPatterns_ana/utils.py`, so a fourth copy here matches
convention rather than breaking it). `masks['NPC']` is the fixed, bilateral
anatomical ROI used throughout (tutorial §3: "one fixed anatomical ROI, same in
both groups").

In [12]:
def load_r2(subject, space, bids_folder, key=NPRF_KEY, session=SESSION):
    """Per-voxel/vertex nPRF R². Returns (r2_flat, ref_img). ref_img is None on surfaces."""
    sub = f'sub-{subject:02d}'
    func = op.join(bids_folder, 'derivatives', key, sub, f'ses-{session}', 'func')

    if space == 'fsaverage5':
        hemis = [nib.load(op.join(
            func, f'{sub}_ses-{session}_desc-r2.optim.nilearn_'
                  f'space-fsaverage5_hemi-{h}.func.gii')).agg_data() for h in ['L', 'R']]
        return np.concatenate(hemis).astype(float), None

    elif space == 'T1w':
        img = nib.load(op.join(
            func, f'{sub}_ses-{session}_desc-r2.optim_space-T1w_pars.nii.gz'))
        return img.get_fdata().ravel().astype(float), img

    raise ValueError(f"space must be 'fsaverage5' or 'T1w', got {space!r}")


def load_masks(subject, space, bids_folder, ref_img=None, session=SESSION):
    """Boolean masks on the same flat index as load_r2. Keys: 'whole brain', 'NPC', 'NPC_L', 'NPC_R'."""
    sub = f'sub-{subject:02d}'

    if space == 'fsaverage5':
        from nilearn import datasets
        atlas = datasets.fetch_atlas_surf_destrieux()
        labeling = np.concatenate([atlas['map_left'], atlas['map_right']])
        drop = [atlas['labels'].index(r) for r in (b'Medial_wall', b'Unknown')]
        cortex = ~np.isin(labeling, drop)

        mdir = op.join(bids_folder, 'derivatives', 'surface_masks')
        npc = {}
        for side, h in [('L', 'lh'), ('R', 'rh')]:
            g = nib.load(op.join(
                mdir, f'desc-NPC_{side}_space-fsaverage5_hemi-{h}.label.gii')).agg_data()
            npc[side] = np.bool_(g)
        zL, zR = np.zeros_like(npc['L']), np.zeros_like(npc['R'])
        npc_L = np.concatenate([npc['L'], zR])
        npc_R = np.concatenate([zL, npc['R']])

    elif space == 'T1w':
        from nilearn.image import resample_to_img, index_img
        bm = nib.load(op.join(
            bids_folder, 'derivatives', 'fmriprep', sub, f'ses-{session}', 'func',
            f'{sub}_ses-{session}_task-magjudge_run-1_space-T1w_desc-brain_mask.nii.gz'))
        cortex = bm.get_fdata().ravel() > 0

        mdir = op.join(bids_folder, 'derivatives', 'ips_masks', sub)
        out = {}
        for side in ['L', 'R']:
            m = nib.load(op.join(mdir, f'{sub}_space-T1w_desc-NPC_{side}.nii.gz'))
            if m.ndim == 4:
                m = index_img(m, 0)
            m = nib.Nifti1Image(m.get_fdata().astype(np.float32), m.affine)
            m = resample_to_img(m, ref_img, interpolation='nearest')
            out[side] = m.get_fdata().ravel() > 0.5
        npc_L, npc_R = out['L'], out['R']

    else:
        raise ValueError(f"space must be 'fsaverage5' or 'T1w', got {space!r}")

    return {'whole brain': cortex,
            'NPC':         (npc_L | npc_R) & cortex,
            'NPC_L':       npc_L & cortex,
            'NPC_R':       npc_R & cortex}

## Group labels, cross-checked

Two independent sources of `group` (0=control, 1=dyscalc) exist in this repo and
are not generated from one another:
- `group_assignment.csv` at the BIDS root (used as-is by `r2_mixture_signal_voxels.ipynb`)
- `add_tables/subjects_recruit_scan_scanned-final.csv`, loaded the same way as
  `numrisk/behavior_general/measures_registry.py::_load_participants` (that module
  hardcodes Maike's Mac path, so the same 5-line loader is reimplemented here
  pointed at `bids_folder` instead of importing it)

Load both, keep only subjects where they agree, and print anything that doesn't
match rather than silently trusting one file.

In [4]:
# source 1: group_assignment.csv (BIDS root)
ga_fn = op.join(bids_folder, 'group_assignment.csv')
ga_df = pd.read_csv(ga_fn).set_index('subject')[['group']]

# source 2: subjects_recruit_scan_scanned-final.csv (add_tables/), same columns/
# coding as measures_registry._load_participants
part_fn = op.join(bids_folder, 'add_tables', 'subjects_recruit_scan_scanned-final.csv')
part_df = pd.read_csv(part_fn, header=0)
part_df = (part_df.loc[:, ['subject ID', 'group']]
                   .rename(columns={'subject ID': 'subject'})
                   .dropna()
                   .astype({'subject': int, 'group': int})
                   .set_index('subject'))

both = ga_df.join(part_df, lsuffix='_ga', rsuffix='_part', how='inner')
mismatch  = both[both['group_ga'] != both['group_part']]
only_ga   = ga_df.index.difference(part_df.index)
only_part = part_df.index.difference(ga_df.index)

print(f'{len(both)} subjects in both sources; {len(mismatch)} disagree')
if len(mismatch):
    print('!! group mismatch -- dropping these subjects:')
    print(mismatch)
if len(only_ga):
    print(f'only in group_assignment.csv: {sorted(only_ga)}')
if len(only_part):
    print(f'only in subjects_recruit_scan_scanned-final.csv: {sorted(only_part)}')

agree = both[both['group_ga'] == both['group_part']]
group_df = agree[['group_ga']].rename(columns={'group_ga': 'group'})
group_df['group_label'] = np.where(group_df['group'] == 0, 'control', 'dyscalc')

print(f'\n{len(group_df)} subjects with an agreeing group label')
group_df['group_label'].value_counts()

66 subjects in both sources; 0 disagree

66 subjects with an agreeing group label


group_label
control    33
dyscalc    33
Name: count, dtype: int64

## Motion / data-quality covariates

`add_tables/QC_framewise_displacement.csv` already exists on the server (produced
by `numrisk/neural_general/check_quality.ipynb`), but nothing else in the repo
currently reads it back in -- joined in fresh here per tutorial §7 ("report mean FD
and n_runs per group in the same table" / "test `noise_mu` between groups first").

In [5]:
qc_fn = op.join(bids_folder, 'add_tables', 'QC_framewise_displacement.csv')
qc_raw = pd.read_csv(qc_fn).set_index('subject')

run_mean_cols = [f'run_{r}_mean' for r in range(1, 7)]
qc_df = pd.DataFrame(index=qc_raw.index)
qc_df['mean_fd'] = qc_raw['overall_mean']
qc_df['n_runs']  = qc_raw[run_mean_cols].notna().sum(axis=1)

print(f'{len(qc_df)} subjects with QC data')
qc_df.describe()

66 subjects with QC data


,mean_fd,n_runs
count,66.000000,66.000000
mean,0.149963,5.939394
std,0.080553,0.240435
min,0.064399,5.000000
25%,0.099254,6.000000
50%,0.140500,6.000000
75%,0.165687,6.000000
max,0.525663,6.000000


## Noise-pinned mixture (tutorial §6.2a)

Overlap between the noise and signal components means `signal_weight` (extent) and
`signal_mu` (strength) can trade off against each other in a single free fit --
about as good a fit comes from "few strong" as from "many weak" vertices, which is
exactly the extent/strength distinction we want to measure (§6). The cheap fix:
each subject's own **whole-brain** fit already estimates the noise component from
tens of thousands of vertices we're confident carry no numerosity tuning, so pin
the ROI fit's `noise_mu`/`noise_sigma` to that whole-brain estimate and only let
`signal_mu`, `signal_sigma` and the weights vary in the ROI fit. Not implemented:
§6.2(b), pinning from a *permutation-null* refit -- skipped for this pass, needs
new PRF model fits.

`braincoder.utils.stats.fit_r2_mixture` doesn't support fixing the noise
component, so this is a small local EM variant -- it reuses the same
logit-transform and weighted-moment-update style `fit_r2_f_beta_mixture` already
uses in that module, just with one component's shape frozen instead of free.

In [13]:
def _logit(r2):
    return np.log(r2 / (1.0 - r2))


def _inv_logit(z):
    return 1.0 / (1.0 + np.exp(-z))


def fit_r2_mixture_noise_pinned(r2, noise_mu, noise_sigma, max_iter=200, tol=1e-7):
    """Logit-Gaussian mixture with the noise component pinned externally.

    noise_mu/noise_sigma are held fixed (e.g. from the subject's own whole-brain
    fit_r2_mixture); only signal_mu, signal_sigma and the weights are updated by EM.
    Returns the same keys as fit_r2_mixture so it's a drop-in for r2_fdr_threshold /
    plot_r2_mixture / posterior_p_signal.
    """
    from scipy.stats import norm
    r2 = np.asarray(r2, dtype=float).ravel()
    r2 = r2[np.isfinite(r2) & (r2 > 0) & (r2 < 0.99)]
    if len(r2) < 50:
        raise ValueError(f'need >= 50 finite R² values in (0, 0.99); got {len(r2)}')
    z = _logit(np.clip(r2, 1e-6, 1 - 1e-6))

    # init the signal component from the upper half of the data
    hi = z[z > np.median(z)]
    signal_mu, signal_sigma = float(hi.mean()), float(max(hi.std(), 1e-3))
    w_n, w_s = 0.9, 0.1

    prev_ll = -np.inf
    for _ in range(max_iter):
        p_n = w_n * norm.pdf(z, noise_mu, noise_sigma)
        p_s = w_s * norm.pdf(z, signal_mu, signal_sigma)
        denom = p_n + p_s + 1e-300
        ll = float(np.log(denom).sum())
        resp_s = p_s / denom
        w_s = float(np.clip(resp_s.mean(), 1e-4, 1 - 1e-4))
        w_n = 1.0 - w_s
        signal_mu = float((resp_s * z).sum() / max(resp_s.sum(), 1e-9))
        signal_sigma = max(float(np.sqrt((resp_s * (z - signal_mu) ** 2).sum()
                                         / max(resp_s.sum(), 1e-9))), 1e-3)
        if abs(ll - prev_ll) < tol:
            break
        prev_ll = ll

    return {
        'mixture':        'gmm_logit_noise_pinned',
        'noise_mu':       float(noise_mu),
        'noise_sigma':    float(noise_sigma),
        'noise_weight':   w_n,
        'noise_mean_r2':  float(_inv_logit(noise_mu)),
        'signal_mu':      signal_mu,
        'signal_sigma':   signal_sigma,
        'signal_weight':  w_s,
        'signal_mean_r2': float(_inv_logit(signal_mu)),
        'log_likelihood': ll,
        'n_voxels':       int(len(r2)),
    }

## Per-subject loop

For each subject with data, an agreeing group label, and a QC row: fit the
whole-brain mixture, the free ROI (NPC) mixture, and the noise-pinned ROI mixture;
compute tutorial §5's guard rails and §5's seed-stability check on the free ROI
fit; record everything (whole-brain `wb_*`, free ROI `roi_*`, pinned `pinned_*`) in
one row per subject.

In [14]:
def _guard_rails(fit):
    delta_mu = fit['signal_mu'] - fit['noise_mu']
    sigma_ratio = fit['signal_sigma'] / fit['noise_sigma']
    return {
        'delta_mu':    delta_mu,
        'sigma_ratio': sigma_ratio,
        # canary -- fit_r2_mixture labels argmax(means)=signal by construction,
        # so this should never actually fire
        'flag_labels_swapped': fit['signal_mu'] <= fit['noise_mu'],
        'flag_ws_gt_wn':       fit['signal_weight'] > fit['noise_weight'],
        'flag_not_separated':  (delta_mu < 0.5) or (sigma_ratio > 1.4),
    }


rows = []
fits_by_subject = {}   # subject -> {'roi_r2', 'fit_wb', 'fit_roi', 'fit_roi_pinned'}

subjects = sorted(set(group_df.index) & set(qc_df.index))
print(f'{len(subjects)} subjects with an agreeing group label + QC row\n')

for sub in subjects:
    try:
        r2_full, ref_img = load_r2(sub, SPACE, bids_folder)
        masks = load_masks(sub, SPACE, bids_folder, ref_img=ref_img)
    except FileNotFoundError as e:
        print(f'sub-{sub:02d}: missing ({op.basename(str(e).split()[-1])})')
        continue

    r2_wb  = r2_full[masks['whole brain']]
    r2_roi = r2_full[masks['NPC']]

    try:
        fit_wb = fit_r2_mixture(r2_wb, n_init=N_INIT, seed=0)
    except ValueError as e:
        print(f'sub-{sub:02d}: whole-brain fit failed ({e})')
        continue
    try:
        fit_roi = fit_r2_mixture(r2_roi, n_init=N_INIT, seed=0)
    except ValueError as e:
        print(f'sub-{sub:02d}: ROI fit failed ({e})')
        continue

    fit_roi_pinned = fit_r2_mixture_noise_pinned(
        r2_roi, noise_mu=fit_wb['noise_mu'], noise_sigma=fit_wb['noise_sigma'])

    guard = _guard_rails(fit_roi)
    thr_fdr = r2_fdr_threshold(fit_roi, alpha=ALPHA)
    noise_tail_r2 = _inv_logit(fit_roi['noise_mu'] + 2 * fit_roi['noise_sigma'])

    w_s_seeds = []
    for seed in SEEDS_STABILITY:
        try:
            w_s_seeds.append(fit_r2_mixture(r2_roi, n_init=N_INIT, seed=seed)['signal_weight'])
        except ValueError:
            pass

    row = {
        'subject':       sub,
        'group':         group_df.loc[sub, 'group'],
        'group_label':   group_df.loc[sub, 'group_label'],
        'mean_fd':       qc_df.loc[sub, 'mean_fd'],
        'n_runs':        qc_df.loc[sub, 'n_runs'],
        'n_vertices_wb':  int(r2_wb.size),
        'n_vertices_roi': int(r2_roi.size),
        **{f'wb_{k}': v for k, v in fit_wb.items() if k != 'mixture'},
        **{f'roi_{k}': v for k, v in fit_roi.items() if k != 'mixture'},
        **{f'pinned_{k}': v for k, v in fit_roi_pinned.items() if k != 'mixture'},
        **{f'roi_{k}': v for k, v in guard.items()},
        'roi_r2_fdr_threshold': thr_fdr,
        'roi_r2_fdr_finite':    bool(np.isfinite(thr_fdr)),
        'roi_noise_tail_r2':    noise_tail_r2,
        'roi_w_s_seed_std':    float(np.std(w_s_seeds)) if w_s_seeds else np.nan,
        'roi_w_s_seed_range':  float(np.ptp(w_s_seeds)) if w_s_seeds else np.nan,
        'roi_w_s_seed_n':      len(w_s_seeds),
    }
    rows.append(row)
    fits_by_subject[sub] = {'roi_r2': r2_roi, 'fit_wb': fit_wb,
                            'fit_roi': fit_roi, 'fit_roi_pinned': fit_roi_pinned}
    print(f'sub-{sub:02d} done', end='\r')

df = pd.DataFrame(rows).set_index('subject')
print(f'\n{len(df)} subjects fit successfully')
df.head()

/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/nilearn/_utils/niimg_conversions.py:80: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  return new_img_like(
/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/nilearn/_utils/niimg_conversions.py:80: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  return new_img_like(


/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/nilearn/_utils/niimg_conversions.py:80: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  return new_img_like(
/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/nilearn/_utils/niimg_conversions.py:80: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  return new_img_like(


/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/nilearn/_utils/niimg_conversions.py:80: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  return new_img_like(
/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/nilearn/_utils/niimg_conversions.py:80: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  return new_img_like(


/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/nilearn/_utils/niimg_conversions.py:80: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  return new_img_like(
/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/nilearn/_utils/niimg_conversions.py:80: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  return new_img_like(


sub-65: missing (sub-65_space-T1w_desc-NPC_L.nii.gz')
sub-66: missing (sub-66_space-T1w_desc-NPC_L.nii.gz')

63 subjects fit successfully


,group,group_label,mean_fd,n_runs,n_vertices_wb,n_vertices_roi,wb_noise_mu,wb_noise_sigma,wb_noise_weight,wb_noise_mean_r2,...,roi_sigma_ratio,roi_flag_labels_swapped,roi_flag_ws_gt_wn,roi_flag_not_separated,roi_r2_fdr_threshold,roi_r2_fdr_finite,roi_noise_tail_r2,roi_w_s_seed_std,roi_w_s_seed_range,roi_w_s_seed_n
subject,,,,,,,,,,,,,,,,,,,,,
1,0,control,0.082219,6,78798,1432,-5.612528,1.159415,0.335390,0.003639,...,0.602171,False,True,False,inf,False,0.034672,0.000000,0.000000,10
2,1,dyscalc,0.472093,6,74165,1482,-5.667165,1.127663,0.337021,0.003446,...,0.563218,False,True,False,inf,False,0.042730,0.000000,0.000000,10
4,1,dyscalc,0.138197,6,82475,1744,-5.568526,1.167524,0.341697,0.003802,...,0.599959,False,True,False,inf,False,0.051863,0.000194,0.000647,10
5,0,control,0.099953,5,80270,1804,-5.557501,1.164708,0.337624,0.003844,...,0.594814,False,True,False,inf,False,0.046610,0.000178,0.000593,10
6,1,dyscalc,0.196975,6,78599,1671,-5.611370,1.179416,0.333737,0.003643,...,0.680313,False,True,False,inf,False,0.038620,0.001515,0.003788,10


## Guard-rail summary

Check these before trusting anything downstream (tutorial §5).

In [15]:
print(f'{len(df)} subjects with a valid ROI fit\n')
print('Guard-rail flags (free ROI fit):')
print(f"  labels swapped (should never fire):              {int(df['roi_flag_labels_swapped'].sum())}")
print(f"  w_s > w_n:                                        {int(df['roi_flag_ws_gt_wn'].sum())}")
print(f"  not separated (delta_mu<0.5 or sigma_ratio>1.4):  {int(df['roi_flag_not_separated'].sum())}")
print(f"  ROI tail-FDR threshold is inf:                    {int((~df['roi_r2_fdr_finite']).sum())}")
print()
print('Seed-stability (std of w_s across seed=0..9), by group:')
df.groupby('group_label')['roi_w_s_seed_std'].describe()[['mean', 'max']]

63 subjects with a valid ROI fit

Guard-rail flags (free ROI fit):
  labels swapped (should never fire):              0
  w_s > w_n:                                        63
  not separated (delta_mu<0.5 or sigma_ratio>1.4):  0
  ROI tail-FDR threshold is inf:                    59

Seed-stability (std of w_s across seed=0..9), by group:


,mean,max
group_label,,
control,0.000803,0.009142
dyscalc,0.000853,0.007454


## Diagnostic plots (PDF)

Tutorial §7: "page all panels into one PDF and actually look at them." Off by
default (`SAVE_DIAGNOSTIC_PDF`), matching the `SAVE_MAP`/`SAVE_GROUP` toggle-cell
convention in `r2_mixture_signal_voxels.ipynb`.

In [18]:
SAVE_DIAGNOSTIC_PDF = True   # flip to True to write one page per subject

if SAVE_DIAGNOSTIC_PDF:
    from matplotlib.backends.backend_pdf import PdfPages

    out_dir = op.join(bids_folder_out, 'derivatives', 'phenotype', 'diagnostics_r2_mixture')
    os.makedirs(out_dir, exist_ok=True)
    pdf_fn = op.join(out_dir,
        f'nPRF_r2mixture_group-dyscalculia_diagnostics_space-{SPACE}_roi-NPC.pdf')

    with PdfPages(pdf_fn) as pdf:
        for sub in df.index:
            info = fits_by_subject[sub]
            thr = r2_fdr_threshold(info['fit_roi'], alpha=ALPHA)
            fig, ax = plt.subplots(figsize=(6, 4))
            plot_r2_mixture(info['fit_roi'], r2=info['roi_r2'], threshold=thr, ax=ax,
                            title=f"sub-{sub:02d} ({df.loc[sub, 'group_label']})  ROI=NPC")
            pdf.savefig(fig)
            plt.close(fig)
    print('wrote', pdf_fn)

wrote /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk/derivatives/phenotype/diagnostics_r2_mixture/nPRF_r2mixture_group-dyscalculia_diagnostics_space-T1w_roi-NPC.pdf


## Group test

Two tests per mixture variant, on the **logit scale** (tutorial §8) --
`sm.formula.ols`, matching this repo's convention in
`behavior_general/across-measur_ana.ipynb` and
`visualize&plots&stats/paper_dnumrisk_01.ipynb` (not the tutorial's literal
`smf.ols`).

`noise_mu ~ group` is tested **first**, on both the whole-brain and free-ROI fits
(tutorial §7): "if [noise_mu] differs, fix the design (match runs/TRs, covary
motion) before interpreting [signal_mu]." 

In [16]:
m_noise_wb  = sm.formula.ols('wb_noise_mu ~ group + mean_fd + n_runs', data=df.reset_index()).fit()
m_noise_roi = sm.formula.ols('roi_noise_mu ~ group + mean_fd + n_runs', data=df.reset_index()).fit()

print('=== whole-brain noise_mu ~ group ===')
print(m_noise_wb.summary())
print('\n=== ROI (free) noise_mu ~ group ===')
print(m_noise_roi.summary())

=== whole-brain noise_mu ~ group ===
                            OLS Regression Results                            
Dep. Variable:            wb_noise_mu   R-squared:                       0.093
Model:                            OLS   Adj. R-squared:                  0.046
Method:                 Least Squares   F-statistic:                     2.005
Date:                Tue, 29 Sep 2026   Prob (F-statistic):              0.123
Time:                        08:46:24   Log-Likelihood:                 69.987
No. Observations:                  63   AIC:                            -132.0
Df Residuals:                      59   BIC:                            -123.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -

### Extent / strength, free ROI fit

In [17]:
df['roi_logit_w'] = np.log(df['roi_signal_weight'] / (1 - df['roi_signal_weight']))

m_w_free  = sm.formula.ols('roi_logit_w ~ group + mean_fd + n_runs + roi_noise_mu',
                            data=df.reset_index()).fit()
m_mu_free = sm.formula.ols('roi_signal_mu ~ group + mean_fd + n_runs + roi_noise_mu',
                            data=df.reset_index()).fit()

print('=== logit(signal_weight) ~ group  [extent] ===')
print(m_w_free.summary())
print('\n=== signal_mu ~ group  [strength] ===')
print(m_mu_free.summary())

=== logit(signal_weight) ~ group  [extent] ===
                            OLS Regression Results                            
Dep. Variable:            roi_logit_w   R-squared:                       0.462
Model:                            OLS   Adj. R-squared:                  0.424
Method:                 Least Squares   F-statistic:                     12.43
Date:                Tue, 29 Sep 2026   Prob (F-statistic):           2.30e-07
Time:                        08:47:13   Log-Likelihood:                 47.232
No. Observations:                  63   AIC:                            -84.46
Df Residuals:                      58   BIC:                            -73.75
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
I

### Extent / strength, noise-pinned ROI fit (§6.2a)

Same two tests on the noise-pinned fit, so the effect of removing the extent/
strength trade-off (rather than the free fit re-estimating noise per ROI) is
visible side by side with the cell above.

In [12]:
df['pinned_logit_w'] = np.log(df['pinned_signal_weight'] / (1 - df['pinned_signal_weight']))

m_w_pinned  = sm.formula.ols('pinned_logit_w ~ group + mean_fd + n_runs + pinned_noise_mu',
                              data=df.reset_index()).fit()
m_mu_pinned = sm.formula.ols('pinned_signal_mu ~ group + mean_fd + n_runs + pinned_noise_mu',
                              data=df.reset_index()).fit()

print('=== logit(signal_weight) ~ group  [extent, noise-pinned] ===')
print(m_w_pinned.summary())
print('\n=== signal_mu ~ group  [strength, noise-pinned] ===')
print(m_mu_pinned.summary())

=== logit(signal_weight) ~ group  [extent, noise-pinned] ===
                            OLS Regression Results                            
Dep. Variable:         pinned_logit_w   R-squared:                       0.460
Model:                            OLS   Adj. R-squared:                  0.424
Method:                 Least Squares   F-statistic:                     12.80
Date:                Mon, 28 Sep 2026   Prob (F-statistic):           1.35e-07
Time:                        15:52:59   Log-Likelihood:                -147.50
No. Observations:                  65   AIC:                             305.0
Df Residuals:                      60   BIC:                             315.9
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------

## Reporting

Per tutorial §9: report `signal_weight`, `signal_mean_r2`, `noise_mean_r2`, and
`delta_mu` per group, for both the free and noise-pinned ROI fits.

In [13]:
summary_cols = ['roi_signal_weight', 'roi_signal_mean_r2', 'roi_noise_mean_r2', 'roi_delta_mu',
                'pinned_signal_weight', 'pinned_signal_mean_r2']
df.groupby('group_label')[summary_cols].agg(['mean', 'std']).round(4)

roi_signal_weight         roi_signal_mean_r2          \
                         mean     std               mean     std   
group_label                                                        
control                0.4417  0.0537             0.0305  0.0107   
dyscalc                0.4489  0.0627             0.0256  0.0100   

            roi_noise_mean_r2         roi_delta_mu          \
                         mean     std         mean     std   
group_label                                                  
control                0.0144  0.0026       0.7313  0.1804   
dyscalc                0.0127  0.0030       0.6783  0.1614   

            pinned_signal_weight         pinned_signal_mean_r2          
                            mean     std                  mean     std  
group_label                                                             
control                   0.8981  0.2339                0.0225  0.0090  
dyscalc                   0.9649  0.1218                0.0175  0.0051

Interpretation (tutorial §8):

| w_s | μ_s | Interpretation |
|---|---|---|
| ↓ | = | **Less area.** Same tuning quality where it exists; smaller numerotopic map. |
| = | ↓ | **Weaker signal.** As much cortex tuned, but the nPRF explains it less well. |
| ↓ | ↓ | Both -- check `noise_mu` hard; a global data-quality difference does exactly this. |
| = | = | The count difference was the noise floor, not tuning. Also a result. |

## Save

In [19]:
SAVE_GROUP = True   # flip to True to write the per-subject table

if SAVE_GROUP:
    out_dir = op.join(bids_folder_out, 'derivatives', 'phenotype')
    os.makedirs(out_dir, exist_ok=True)
    out_fn = op.join(out_dir, f'nPRF_r2mixture_group-dyscalculia_space-{SPACE}_roi-NPC.csv')
    df.to_csv(out_fn)
    print('wrote', out_fn)

wrote /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk/derivatives/phenotype/nPRF_r2mixture_group-dyscalculia_space-T1w_roi-NPC.csv


## Not implemented in this pass

- **§6.2(b) permutation-null noise anchor** -- refit the nPRF on permuted
  trial→numerosity labels (~10-20 permutations/subject) and take `noise_mu`/
  `noise_sigma` from the resulting null R² distribution. Only needed if §6.2(a)
  (already implemented above) still leaves w_s/μ_s trading off.
- **§6.1 bootstrap vertex-resampling CIs** -- 500 within-subject resamples to get
  CIs on w_s/μ_s and check their identifiability directly.
- **§7 leave-one-run-out cvR² cross-check** -- count vertices where
  cvR²(nPRF) > cvR²(null); immune to the in-sample overfitting bias that inflates
  `noise_mu`.

All three need new PRF model fits (`ParameterFitter` + `get_rsq`), not just
statistics on the R² maps already on disk -- a separate, heavier follow-up.